In [1]:
from dotenv import load_dotenv
import os

from langchain_core.messages import HumanMessage
from sympy import false

# 使用绝对路径加载 .env 文件
env_path = r'D:\develop\Pythonai\Langchain\.env'
load_dotenv(env_path)

# 强制使用 127.0.0.1 而非 localhost（避免代理问题）
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'

print("环境变量加载成功")

环境变量加载成功


In [3]:
from langchain.tools import tool

@tool
def get_current_weather(location: str) -> str:
    """Get the current weather in a given location"""
    return f"Current weather in {location} is sunny"

In [4]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

# 创建 Ollama LLM 实例 - 使用 127.0.0.1 和可用模型
llm = init_chat_model(
    model="gemma4:e4b",  # 使用可用的模型
    model_provider="ollama",
    base_url="http://127.0.0.1:11434",  # 关键：使用 127.0.0.1
    temperature=0
)

# 创建 agent
agent = create_agent(
    model=llm,
    tools=[get_current_weather],
    debug=False
)
print("Agent 创建成功")

Agent 创建成功


In [9]:
print('调用大模型发送请求')

# 正确的调用方式
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "今天长沙天气如何？"}
    ]
})

# 打印响应
for message in response["messages"]:
    print(f"角色: {message.type}")
    print(f"内容: {message.content}")
    print("-" * 50)

调用大模型发送请求


ConnectError: [WinError 10061] 由于目标计算机积极拒绝，无法连接。

In [5]:
import os
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_tavily import TavilySearch
base_url = os.getenv("DASHSCOPE_BASE_URL")
api_key = os.getenv("DASHSCOPE_API_KEY")

search_tool = TavilySearch(api_key=os.getenv("TAVILY_API_KEY"),
                           base_url=os.getenv("TAVILY_BASE_URL"),
                           max_results=5,
                           timeout=30,
                           topic = "general")


model = init_chat_model(
    model="qwen3-max",
    model_provider="openai",
    base_url=base_url,
    api_key=api_key,
    temperature=1.5,
    max_tokens=1024,
    top_p=0.9
)


In [6]:
agent = create_agent(
    model=model,
    tools=[get_current_weather, search_tool],
    debug=False
)

In [8]:
stream =agent.stream(
    {"messages" :[HumanMessage(content="蚌埠住了是什么梗？"),]},
    stream_mode = "messages"
    )
for chunk,metadata in stream:
    if chunk.content :
        print(chunk.content, end="", flush=True)

{"query": "蚌埠住了是什么梗", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://zhuanlan.zhihu.com/p/400277735", "title": "【每日一梗】第二十一期蚌埠住了", "content": "读音是{bèng bù} 【蚌埠住了】 谐音“绷不住了” 蚌埠是地名，隶属安徽省地级市，作为一个谐音梗最开始常出现贴吧常用来形容自己情感上受到较大冲击，快撑不住", "score": 0.8793233, "raw_content": null}, {"url": "https://baike.baidu.com/item/%E8%9A%8C%E5%9F%A0%E4%BD%8F%E4%BA%86/55807298", "title": "蚌埠住了_百度百科", "content": "# 蚌埠住了. 蚌埠住了，网络流行语，拼音为bèng bù zhù le，是“绷不住了”的谐音形式，外文译作“I can't hold it”。 [1]. 该词用来形容情感受到强烈冲击难以自持的状态，既可指因搞笑而大笑，也可指因崩溃事件产生无奈或哭泣反应，情绪指向由具体语境决定。 [1]. 词语来源结合安徽蚌埠方言发音与“绷不住”的谐音，早期在贴吧社区传播，抗压背锅吧等网络社群的讨论推动了其流行。2020年4月，一段名为《蚌埠住了，蚌埠住了》的视频加速了该词的传播。 [1]. :   I can't hold it. ## 目录. ## 词语来源. 谐音梗与方言融合：“蚌埠住了” 是 “绷不住了” 的谐音。一方面，安徽蚌埠方言中 “蚌埠住” 有 “撑不住”“坚持不住” 的意思，这种方言说法为其提供了一定的语言基础。另一方面，“蚌埠” 的拼音发音与 “绷不住” 相似，作为谐音梗被网友广泛使用。. 早期传播平台：该词最早在贴吧，尤其是抗压背锅吧等社区广泛流传。2020 年 4 月有一段名为《蚌埠住了，蚌埠住了》的视频，视频中一位蚌埠市民因疫情封控在家无法出门，萌生了做沙雕视频的想法，并配上了这句 “蚌埠住了”，也推动了该词的传播。 [1]. ## 引申含义. 主要用来形容一个人情感上受到较大冲击，快撑不住了，可能是要哭、要笑了或者

KeyboardInterrupt: 

In [12]:
from langchain_core.messages import SystemMessage
from langgraph.checkpoint.memory import InMemorySaver
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
# langchain提供的checkpointer的默认实现，基于内存存储

# 初始化checkpointer
checkpointer = SqliteSaver(sqlite3.connect("D:/develop/Pythonai/resources/checkpoint.db", check_same_thread=False))
# 自动建表
checkpointer.setup()


agent = create_agent(
    model=model,
    tools=[get_current_weather, search_tool],
    system_prompt="你是一个有帮助的助手。使用搜索工具获取信息后,请用简洁自然的语言总结答案,不要直接输出原始的JSON数据。",
    debug=False,
    checkpointer=checkpointer
)

config = {"configurable": {"thread_id": "thread_2"}}
# 然后正常调用
response = agent.invoke(
    {"messages": [HumanMessage(content="我喜欢吃西红柿炒蛋")]},
    config,
)

for message in response["messages"]:
    if message.type == "ai":
        print(message.content)


西红柿炒蛋确实是一道经典又美味的家常菜！酸甜的西红柿搭配嫩滑的鸡蛋，简单却很下饭。你平时做这道菜有什么特别的做法或小秘诀吗？比如加糖还是加番茄酱？


In [13]:
response = agent.invoke(
    {"messages": [HumanMessage(content="我喜欢吃什么来着？")]},
    config,
)

for message in response["messages"]:
    if message.type == "ai":
        print(message.content)


西红柿炒蛋确实是一道经典又美味的家常菜！酸甜的西红柿搭配嫩滑的鸡蛋，简单却很下饭。你平时做这道菜有什么特别的做法或小秘诀吗？比如加糖还是加番茄酱？
你刚刚提到喜欢吃**西红柿炒蛋**！
